---

# Inference & Evaluation Pipeline

**Context-aware autoregressive inference:**
1. Page image → BN_DRISHTI segmentation → line dirs → word crops
2. Word 1 (no context) → **isolated path**: `[patches(128) | BOS, ...]`
3. Word 2+ → **context-aware path**: `[ctx(≤396) | SEP₁ | patches(128) | SEP₂ | BOS, ...]`
4. Context accumulates across words AND lines (not reset at line boundaries)
5. When context exceeds 396-token budget, keep most recent tokens (left-truncation)

**Evaluation:** Predicted text compared against ground truth using CER and WER.

In [ ]:
# ================================================================
# CELL 14 — INFERENCE SETUP: BN_DRISHTI + CHECKPOINT DOWNLOAD
# ================================================================

import sys, os, shutil, time, glob
import torch
import torch.nn.functional as F
from PIL import Image

# --- Configuration ---
# Google Drive checkpoint path (adjust to your actual path)
CHECKPOINT_DRIVE_PATH = '/content/drive/MyDrive/context_aware_v2_checkpoints/checkpoint_epoch_22.pth'
CHECKPOINT_LOCAL_PATH = '/content/checkpoint_epoch_22.pth'

# Test data paths (adjust to your Google Drive folder)
TEST_IMAGES_DIR    = '/content/test_data/images/'    # folder of page images (.jpg/.png)
GROUND_TRUTH_FILE  = '/content/test_data/ground_truth.txt'  # one page transcription per file, or single file

# Inference settings
INF_NUM_BEAMS      = 3       # 1 = greedy, >1 = beam search
INF_MAX_NEW_TOKENS = 100     # max tokens per word (TEXT_TOKENS - 2 for BOS/EOS headroom)
INF_DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

# BN_DRISHTI paths
REPO_ROOT = '/content/GraDeT-HTR'  # adjust if different
BN_DRISHTI_DIR   = os.path.join(REPO_ROOT, 'BN_DRISHTI')
YOLOV5_DIR       = os.path.join(BN_DRISHTI_DIR, 'yolov5')
LINE_WEIGHTS     = os.path.join(BN_DRISHTI_DIR, 'model_weights', 'line_model_best.pt')
WORD_WEIGHTS     = os.path.join(BN_DRISHTI_DIR, 'model_weights', 'word_model_best.pt')

SEGMENTATION_OUTPUT_DIR = '/content/inference_segmentations'
PREDICTION_OUTPUT_DIR   = '/content/inference_predictions'

# --- Mount Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    HAS_DRIVE_INF = True
except ImportError:
    print('Not on Colab — skipping Drive mount.')
    HAS_DRIVE_INF = False

# --- Download checkpoint to local ---
if os.path.exists(CHECKPOINT_DRIVE_PATH) and not os.path.exists(CHECKPOINT_LOCAL_PATH):
    print(f'Copying checkpoint from Drive...')
    shutil.copy2(CHECKPOINT_DRIVE_PATH, CHECKPOINT_LOCAL_PATH)
    print(f'  → {CHECKPOINT_LOCAL_PATH} ({os.path.getsize(CHECKPOINT_LOCAL_PATH)/1e6:.1f} MB)')
elif os.path.exists(CHECKPOINT_LOCAL_PATH):
    print(f'Checkpoint already at {CHECKPOINT_LOCAL_PATH}')
else:
    print(f'WARNING: Checkpoint not found at {CHECKPOINT_DRIVE_PATH}')

# --- Setup BN_DRISHTI / YOLOv5 ---
if not os.path.isdir(YOLOV5_DIR):
    print('Setting up YOLOv5...')
    os.makedirs(BN_DRISHTI_DIR, exist_ok=True)
    !cd {BN_DRISHTI_DIR} && git clone https://github.com/ultralytics/yolov5
    # Copy custom detect.py
    custom_detect = os.path.join(BN_DRISHTI_DIR, 'detect.py')
    if os.path.exists(custom_detect):
        shutil.copy2(custom_detect, os.path.join(YOLOV5_DIR, 'detect.py'))
        print('  Copied custom detect.py into yolov5/')
else:
    print(f'YOLOv5 already set up at {YOLOV5_DIR}')

# Download detection weights if missing
os.makedirs(os.path.join(BN_DRISHTI_DIR, 'model_weights'), exist_ok=True)
if not os.path.exists(LINE_WEIGHTS):
    print('Downloading line detection model...')
    !wget -q -P {os.path.join(BN_DRISHTI_DIR, 'model_weights')} https://huggingface.co/crusnic/BN-DRISHTI/resolve/main/models/line_model_best.pt
if not os.path.exists(WORD_WEIGHTS):
    print('Downloading word detection model...')
    !wget -q -P {os.path.join(BN_DRISHTI_DIR, 'model_weights')} https://huggingface.co/crusnic/BN-DRISHTI/resolve/main/models/word_model_best.pt

# Add repo to path for imports
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Ensure BN_DRISHTI content dir exists
os.makedirs(os.path.join(BN_DRISHTI_DIR, 'content'), exist_ok=True)

print('Inference setup complete.')

In [ ]:
# ================================================================
# CELL 15 — AUTOREGRESSIVE GENERATION (Greedy + Beam Search)
# Handles both isolated and context-aware paths.
# BOS=EOS workaround: skip EOS check on first generated token.
# ================================================================

import torch
import torch.nn.functional as F


@torch.no_grad()
def generate_greedy(
    model,
    pixel_values,          # [1, C, H, W]
    bos_id,
    eos_id,
    context_ids=None,      # [1, ctx_len] or None (isolated)
    context_mask=None,     # [1, ctx_len] or None
    max_new_tokens=100,
    device='cpu',
):
    """Greedy autoregressive decode with KV cache."""
    input_ids = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    attn_mask = torch.ones(1, 1, dtype=torch.long, device=device)
    past_kv   = None
    generated = []

    for step in range(max_new_tokens):
        out = model(
            input_ids      = input_ids,
            pixel_values   = pixel_values if past_kv is None else None,
            context_ids    = context_ids  if past_kv is None else None,
            context_mask   = context_mask if past_kv is None else None,
            attention_mask = attn_mask,
            past_key_values = past_kv,
            use_cache      = True,
        )
        next_token = out.logits[:, -1, :].argmax(dim=-1).item()

        # BOS=EOS guard: don't stop on the very first generated token
        if next_token == eos_id and step > 0:
            break

        generated.append(next_token)
        input_ids = torch.tensor([[next_token]], dtype=torch.long, device=device)
        attn_mask = torch.ones(1, 1, dtype=torch.long, device=device)
        past_kv   = out.past_key_values

    return generated


@torch.no_grad()
def generate_beam_search(
    model,
    pixel_values,          # [1, C, H, W]
    bos_id,
    eos_id,
    context_ids=None,
    context_mask=None,
    num_beams=3,
    max_new_tokens=100,
    device='cpu',
):
    """Beam search with KV cache. Returns best beam's token list."""
    input_ids = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    attn_mask = torch.ones(1, 1, dtype=torch.long, device=device)

    # --- First forward pass (prefill) ---
    out = model(
        input_ids      = input_ids,
        pixel_values   = pixel_values,
        context_ids    = context_ids,
        context_mask   = context_mask,
        attention_mask = attn_mask,
        use_cache      = True,
    )
    logits = out.logits[:, -1, :]  # [1, V]

    # Mask out EOS on first step (BOS=EOS workaround)
    logits[:, eos_id] = float('-inf')
    log_probs = F.log_softmax(logits, dim=-1)

    topk_scores, topk_ids = log_probs.topk(num_beams, dim=-1)  # [1, B]
    beam_scores = topk_scores.squeeze(0)    # [B]
    beam_tokens = topk_ids.squeeze(0).unsqueeze(-1)  # [B, 1]

    # Expand KV cache: [1, heads, seq, dim] -> [B, heads, seq, dim]
    past_kv = tuple(
        (k.expand(num_beams, -1, -1, -1).contiguous(),
         v.expand(num_beams, -1, -1, -1).contiguous())
        for k, v in out.past_key_values
    )

    done_mask = torch.zeros(num_beams, dtype=torch.bool, device=device)
    # Track finished beams and their scores
    finished_beams = []  # list of (score, token_sequence)

    for step in range(1, max_new_tokens):
        if done_mask.all():
            break

        # Next input: last token of each beam
        next_input = beam_tokens[:, -1:].to(device)  # [B, 1]
        next_attn  = torch.ones(num_beams, 1, dtype=torch.long, device=device)

        out = model(
            input_ids       = next_input,
            attention_mask  = next_attn,
            past_key_values = past_kv,
            use_cache       = True,
        )
        logits = out.logits[:, -1, :]  # [B, V]
        log_probs = F.log_softmax(logits, dim=-1)

        # Zero out scores for already-finished beams (only allow EOS/pad)
        for i in range(num_beams):
            if done_mask[i]:
                log_probs[i, :] = float('-inf')
                log_probs[i, eos_id] = 0.0

        # Expand: [B] + [B, V] -> [B, V]
        next_scores = beam_scores.unsqueeze(-1) + log_probs
        V = log_probs.shape[-1]

        # Pick top num_beams across all beams × vocab
        flat = next_scores.view(-1)
        topk_flat_scores, topk_flat_idx = flat.topk(num_beams, dim=-1)

        beam_idx  = topk_flat_idx // V   # which source beam
        token_idx = topk_flat_idx % V    # which token

        # Reorder sequences and scores
        beam_tokens = torch.cat([beam_tokens[beam_idx], token_idx.unsqueeze(-1)], dim=-1)
        beam_scores = topk_flat_scores

        # Reorder KV cache to match new beam ordering
        past_kv = tuple(
            (k[beam_idx], v[beam_idx])
            for k, v in out.past_key_values
        )

        # Reorder done mask
        done_mask = done_mask[beam_idx]

        # Check for newly finished beams
        for i in range(num_beams):
            if token_idx[i].item() == eos_id and not done_mask[i]:
                done_mask[i] = True
                # Length-normalize score
                length = beam_tokens.shape[1]
                norm_score = beam_scores[i].item() / length
                finished_beams.append((norm_score, beam_tokens[i].tolist()))

    # If no beam finished, take the best active beam
    if not finished_beams:
        best_idx = beam_scores.argmax().item()
        return beam_tokens[best_idx].tolist()

    # Return the beam with highest normalized score
    finished_beams.sort(key=lambda x: x[0], reverse=True)
    return finished_beams[0][1]


def generate(
    model,
    processor,
    pixel_values,
    context_text=None,
    num_beams=1,
    max_new_tokens=100,
    device='cpu',
):
    """
    High-level generation interface.
    - If context_text is provided and non-empty: context-aware path
    - Otherwise: isolated path (no context, no SEP tokens)

    Returns decoded string.
    """
    bos_id = processor.tokeniser.bos_token_id
    eos_id = processor.tokeniser.eos_token_id

    # Encode context if provided
    ctx_ids, ctx_mask = None, None
    if context_text and context_text.strip():
        ctx_ids, ctx_mask = processor._encode_context(context_text)
        ctx_ids  = ctx_ids.to(device)
        ctx_mask = ctx_mask.to(device)

    pv = pixel_values.to(device)
    if pv.dim() == 3:
        pv = pv.unsqueeze(0)  # [C,H,W] -> [1,C,H,W]

    if num_beams <= 1:
        token_ids = generate_greedy(
            model, pv, bos_id, eos_id,
            context_ids=ctx_ids, context_mask=ctx_mask,
            max_new_tokens=max_new_tokens, device=device,
        )
    else:
        token_ids = generate_beam_search(
            model, pv, bos_id, eos_id,
            context_ids=ctx_ids, context_mask=ctx_mask,
            num_beams=num_beams, max_new_tokens=max_new_tokens,
            device=device,
        )

    # Decode and clean
    text = processor.tokeniser.decode(token_ids, skip_special_tokens=True)
    return text.strip()


print('Generation functions defined: generate_greedy, generate_beam_search, generate')

In [ ]:
# ================================================================
# CELL 16 — CONTEXT-AWARE PAGE INFERENCE + SEGMENTATION HELPERS
# ================================================================

from segment_single_page import (
    load_segmentation_models, run_segmentation_model, clean_workspace
)


def sort_underscore_numbers(keys):
    """Sort strings like '1_1_2', '1_1_10' by numeric components."""
    return sorted(keys, key=lambda s: tuple(int(p) for p in os.path.splitext(s)[0].split('_')))


def segment_page(image_path, line_model_cfg, word_model_cfg, output_dir, root_path=''):
    """
    Run BN_DRISHTI 3-pass segmentation on a single page image.
    Returns path to the directory containing line subdirectories with word crops.
    """
    image_label = os.path.splitext(os.path.basename(image_path))[0]
    page_seg_dir = os.path.join(output_dir, image_label) + '/'

    # Clean previous segmentation artifacts
    clean_workspace(root_path=root_path)
    os.makedirs(page_seg_dir, exist_ok=True)

    run_segmentation_model(
        image_path=image_path,
        image_label=image_label,
        line_model_config=line_model_cfg,
        word_model_config=word_model_cfg,
        final_word_segmentation=page_seg_dir,
        root_path=root_path,
    )
    return page_seg_dir


def infer_page_context_aware(
    page_seg_dir,
    model,
    processor,
    num_beams=3,
    max_new_tokens=100,
    device='cpu',
):
    """
    Context-aware inference on a segmented page.

    Flow:
    - Word 1 (no prior context) → isolated path
    - Word 2+ → context-aware path with accumulated predictions
    - Context accumulates across words AND across lines (no reset)
    - When context exceeds 396-token budget, processor keeps most recent tokens

    Returns:
        predicted_text (str): full page prediction, lines joined by newline
        word_timings (list): per-word timing info for speed analysis
    """
    # Find line subdirectories
    subdirs = [d for d in os.listdir(page_seg_dir)
               if os.path.isdir(os.path.join(page_seg_dir, d))]
    if not subdirs:
        print(f'  WARNING: No line directories found in {page_seg_dir}')
        return '', []

    sorted_lines = sort_underscore_numbers(subdirs)

    all_lines = []
    accumulated_context = ''   # carries across lines
    word_timings = []
    total_words = 0

    for line_name in sorted_lines:
        line_dir = os.path.join(page_seg_dir, line_name)
        files = [f for f in os.listdir(line_dir)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not files:
            continue

        base2file = {os.path.splitext(f)[0]: f for f in files}
        sorted_bases = sort_underscore_numbers(list(base2file.keys()))

        line_words = []
        for base in sorted_bases:
            img_path = os.path.join(line_dir, base2file[base])
            image = Image.open(img_path).convert('RGB')

            # Process image through ViT processor
            img_out = processor.vit_processor(
                image,
                input_data_format='channels_last',
                return_tensors='pt',
            )
            pv = img_out['pixel_values']
            if not isinstance(pv, torch.Tensor):
                pv = torch.tensor(pv, dtype=torch.float32)

            # Determine path: isolated (no context) vs context-aware
            ctx = accumulated_context if accumulated_context.strip() else None

            t0 = time.time()
            word_text = generate(
                model, processor, pv,
                context_text=ctx,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                device=device,
            )
            elapsed = time.time() - t0

            total_words += 1
            path_type = 'ctx' if ctx else 'iso'
            word_timings.append({
                'word_idx': total_words,
                'line': line_name,
                'file': base,
                'path': path_type,
                'predicted': word_text,
                'time_sec': elapsed,
            })

            line_words.append(word_text)

            # Accumulate context: append predicted word
            if accumulated_context:
                accumulated_context = accumulated_context + ' ' + word_text
            else:
                accumulated_context = word_text

        all_lines.append(' '.join(line_words))

    predicted_text = '\n'.join(all_lines)
    return predicted_text, word_timings


print('Page inference functions defined: segment_page, infer_page_context_aware')

In [ ]:
# ================================================================
# CELL 17 — EVALUATION METRICS (CER, WER)
# ================================================================


def _edit_distance(ref, hyp):
    """Standard Levenshtein edit distance between two sequences."""
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[n][m]


def compute_cer(reference: str, hypothesis: str) -> float:
    """Character Error Rate = edit_distance(ref_chars, hyp_chars) / len(ref_chars)."""
    ref_chars = list(reference)
    hyp_chars = list(hypothesis)
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    return _edit_distance(ref_chars, hyp_chars) / len(ref_chars)


def compute_wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = edit_distance(ref_words, hyp_words) / len(ref_words)."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return _edit_distance(ref_words, hyp_words) / len(ref_words)


def evaluate_predictions(predictions: dict, ground_truths: dict):
    """
    Compare predictions against ground truths.

    Args:
        predictions:   {page_name: predicted_text}
        ground_truths: {page_name: reference_text}

    Returns dict with per-page and aggregate metrics.
    """
    results = {}
    total_cer, total_wer, count = 0.0, 0.0, 0

    for page_name in sorted(predictions.keys()):
        pred = predictions[page_name]
        if page_name not in ground_truths:
            print(f'  WARNING: No ground truth for page "{page_name}", skipping evaluation.')
            results[page_name] = {'predicted': pred, 'reference': None, 'cer': None, 'wer': None}
            continue

        ref = ground_truths[page_name]
        cer = compute_cer(ref, pred)
        wer = compute_wer(ref, pred)
        total_cer += cer
        total_wer += wer
        count += 1

        results[page_name] = {
            'predicted': pred,
            'reference': ref,
            'cer': cer,
            'wer': wer,
        }

    avg_cer = total_cer / count if count > 0 else float('nan')
    avg_wer = total_wer / count if count > 0 else float('nan')

    return {
        'per_page': results,
        'avg_cer': avg_cer,
        'avg_wer': avg_wer,
        'num_evaluated': count,
    }


def load_ground_truths(gt_path):
    """
    Load ground truth from a file or directory.

    Supported formats:
    (A) Single .txt file with sections separated by '=== page_name ===':
        === 1_1 ===
        আমি বাংলায় গান গাই
        === 1_2 ===
        ...

    (B) Directory of .txt files (one per page, filename = page_name.txt):
        ground_truth/
          1_1.txt  →  contains transcription for page 1_1
          1_2.txt  →  ...
    """
    gt = {}

    if os.path.isdir(gt_path):
        for f in os.listdir(gt_path):
            if f.endswith('.txt'):
                page_name = os.path.splitext(f)[0]
                with open(os.path.join(gt_path, f), 'r', encoding='utf-8') as fh:
                    gt[page_name] = fh.read().strip()
    elif os.path.isfile(gt_path):
        with open(gt_path, 'r', encoding='utf-8') as fh:
            content = fh.read()

        # Try sectioned format first
        if '===' in content:
            current_page = None
            current_lines = []
            for line in content.split('\n'):
                stripped = line.strip()
                if stripped.startswith('===') and stripped.endswith('==='):
                    if current_page is not None:
                        gt[current_page] = '\n'.join(current_lines).strip()
                    current_page = stripped.strip('= ').strip()
                    current_lines = []
                else:
                    if current_page is not None:
                        current_lines.append(line)
            if current_page is not None:
                gt[current_page] = '\n'.join(current_lines).strip()
        else:
            # Single page: use filename as page name
            page_name = os.path.splitext(os.path.basename(gt_path))[0]
            gt[page_name] = content.strip()
    else:
        print(f'WARNING: Ground truth path not found: {gt_path}')

    return gt


print(f'Evaluation functions defined: compute_cer, compute_wer, evaluate_predictions, load_ground_truths')

In [ ]:
# ================================================================
# CELL 18 — RUN FULL INFERENCE + EVALUATION PIPELINE
# ================================================================

print('=' * 70)
print('CONTEXT-AWARE INFERENCE & EVALUATION PIPELINE')
print('=' * 70)

# ---- 1. Load DTrOCR model from checkpoint ----
print('\n[1/4] Loading DTrOCR model...')
cfg_inf = DTrOCRConfig()
model_inf = DTrOCRLMHeadModel(cfg_inf)

# Load from checkpoint (has 'model_state_dict' key, not bare state_dict)
ckpt = torch.load(CHECKPOINT_LOCAL_PATH, map_location='cpu', weights_only=True)
if 'model_state_dict' in ckpt:
    sd = ckpt['model_state_dict']
else:
    sd = ckpt
# Strip torch.compile prefix if present
sd = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v for k, v in sd.items()}
model_inf.load_state_dict(sd, strict=False)
model_inf.to(INF_DEVICE)
model_inf.eval()

# Processor for inference: add_bos_token=False to avoid double-BOS
processor_inf = DTrOCRProcessor(cfg_inf, add_bos_token=False, add_eos_token=False)

epoch_loaded = ckpt.get('epoch', '?')
print(f'  Model loaded from {CHECKPOINT_LOCAL_PATH} (epoch {epoch_loaded})')
print(f'  Device: {INF_DEVICE}')
print(f'  Vocab: {cfg_inf.vocab_size}, Ctx budget: {cfg_inf.max_context_length} tokens')
print(f'  Beams: {INF_NUM_BEAMS}, Max new tokens/word: {INF_MAX_NEW_TOKENS}')

# ---- 2. Load BN_DRISHTI segmentation models ----
print('\n[2/4] Loading segmentation models...')
line_model_cfg, word_model_cfg = load_segmentation_models(
    line_weights=LINE_WEIGHTS,
    word_weights=WORD_WEIGHTS,
    half=(INF_DEVICE != 'cpu'),
    device=INF_DEVICE,
)
print('  Line + word detection models loaded.')

# ---- 3. Run inference on all test page images ----
print(f'\n[3/4] Running inference on pages in: {TEST_IMAGES_DIR}')

# Gather page images
page_images = sorted([
    f for f in os.listdir(TEST_IMAGES_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])
print(f'  Found {len(page_images)} page image(s)')

os.makedirs(SEGMENTATION_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

predictions = {}          # {page_name: predicted_text}
all_word_timings = []     # timing data for speed analysis
total_start = time.time()

for img_file in page_images:
    page_name = os.path.splitext(img_file)[0]
    img_path  = os.path.join(TEST_IMAGES_DIR, img_file)
    print(f'\n  --- Page: {page_name} ---')

    # Step A: Segment
    t_seg = time.time()
    page_seg_dir = segment_page(
        img_path, line_model_cfg, word_model_cfg,
        output_dir=SEGMENTATION_OUTPUT_DIR,
        root_path=REPO_ROOT + '/',
    )
    seg_time = time.time() - t_seg
    print(f'  Segmentation: {seg_time:.1f}s')

    # Step B: Context-aware inference
    t_inf = time.time()
    pred_text, word_timings = infer_page_context_aware(
        page_seg_dir, model_inf, processor_inf,
        num_beams=INF_NUM_BEAMS,
        max_new_tokens=INF_MAX_NEW_TOKENS,
        device=INF_DEVICE,
    )
    inf_time = time.time() - t_inf

    predictions[page_name] = pred_text
    all_word_timings.extend(word_timings)

    n_words = len(word_timings)
    ctx_words = sum(1 for w in word_timings if w['path'] == 'ctx')
    iso_words = n_words - ctx_words
    print(f'  Inference: {inf_time:.1f}s  |  {n_words} words ({iso_words} isolated, {ctx_words} context-aware)')

    # Save prediction to file
    pred_path = os.path.join(PREDICTION_OUTPUT_DIR, f'{page_name}.txt')
    with open(pred_path, 'w', encoding='utf-8') as f:
        f.write(pred_text)

    # Print prediction
    print(f'\n  PREDICTED TEXT:')
    for line in pred_text.split('\n'):
        print(f'    {line}')

total_time = time.time() - total_start
total_words = len(all_word_timings)
print(f'\n  Total pipeline time: {total_time:.1f}s for {len(page_images)} pages, {total_words} words')
if total_words > 0:
    avg_per_word = sum(w['time_sec'] for w in all_word_timings) / total_words
    print(f'  Avg inference time per word: {avg_per_word*1000:.0f}ms')

# ---- 4. Evaluate against ground truth ----
print(f'\n[4/4] Evaluation against ground truth...')

gt = load_ground_truths(GROUND_TRUTH_FILE)
if not gt:
    print(f'  No ground truth found at {GROUND_TRUTH_FILE}')
    print('  Skipping evaluation. Set GROUND_TRUTH_FILE to a valid path.')
else:
    print(f'  Loaded ground truth for {len(gt)} page(s): {list(gt.keys())}')
    eval_results = evaluate_predictions(predictions, gt)

    print(f'\n{"=" * 70}')
    print(f'RESULTS  (evaluated on {eval_results["num_evaluated"]} pages)')
    print(f'{"=" * 70}')
    print(f'  Average CER: {eval_results["avg_cer"]:.4f}  ({eval_results["avg_cer"]*100:.2f}%)')
    print(f'  Average WER: {eval_results["avg_wer"]:.4f}  ({eval_results["avg_wer"]*100:.2f}%)')

    # Per-page side-by-side comparison
    for page_name, res in eval_results['per_page'].items():
        if res['reference'] is None:
            continue
        print(f'\n  --- {page_name} --- CER: {res["cer"]:.4f}  WER: {res["wer"]:.4f}')

        ref_lines = res['reference'].split('\n')
        pred_lines = res['predicted'].split('\n')
        max_lines = max(len(ref_lines), len(pred_lines))

        print(f'  {"GROUND TRUTH":<50s} | {"PREDICTION":<50s}')
        print(f'  {"-"*50} | {"-"*50}')
        for i in range(max_lines):
            r = ref_lines[i] if i < len(ref_lines) else ''
            p = pred_lines[i] if i < len(pred_lines) else ''
            match = 'OK' if r == p else 'XX'
            print(f'  {r:<50s} | {p:<50s}  [{match}]')

print(f'\nPredictions saved to: {PREDICTION_OUTPUT_DIR}/')
print('Done.')

# Context-Aware DTrOCR — Context-Only Training

**Simplified training setup:**
- Context-only training (no dual-path, no isolated path)
- Default loss function: `L = L_ctx` (standard cross-entropy)
- Configurable `ARTICLE_BARRIER` — set True to keep context within articles, False for cross-document
- Budget-fill context (token-level, no word-count limit — gobbles up as much as budget allows)
- `[SEP]` tokens between modalities: `[ctx(396) | SEP₁ | patches(128) | SEP₂ | text(124)]`
- 50,257 Bengali GPT-2 BPE tokenizer (flax-community/gpt2-bengali)
- Position embedding extended to 654 to fix generation OOB crash
- SDPA attention mask used consistently
- Pre-trained Bengali GPT-2 weights loaded into transformer blocks + token embeddings
- `batch_size=16`, default cross-entropy loss
- Padding labels set to `-100` (CrossEntropyLoss ignore index)
- `weights_only=True` in all `torch.load` calls
- Optimizer state moved to correct device after checkpoint load
- Context-only validation during training
- Context cache precomputed in `__init__`
- NaN counter reset per epoch

In [ ]:
# ================================================================
# CELL 0 — GOOGLE DRIVE SETUP
# ================================================================

!pip install -q gdown

import os

GDRIVE_URL  = 'https://drive.google.com/file/d/1txYG1JoP62RWWrn_0FxCQ92g9pvYNFWl/view?usp=drive_link'
ZIP_DEST    = '/content/dataset.zip'
EXTRACT_DIR = '/content/data'

if not os.path.exists(ZIP_DEST):
    !gdown --fuzzy "{GDRIVE_URL}" -O "{ZIP_DEST}"
else:
    print(f'Already downloaded: {ZIP_DEST}')

if not os.path.exists(ZIP_DEST) or os.path.getsize(ZIP_DEST) < 1000:
    print('ERROR: Download failed or file too small.')
else:
    print(f'Downloaded: {ZIP_DEST} ({os.path.getsize(ZIP_DEST) / 1e6:.1f} MB)')
    if not os.path.exists(EXTRACT_DIR):
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        !unzip -qo "{ZIP_DEST}" -d "{EXTRACT_DIR}"
    print(f'Extracted to: {EXTRACT_DIR}')
    for root, dirs, files in os.walk(EXTRACT_DIR):
        depth = root.replace(EXTRACT_DIR, '').count(os.sep)
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth < 2:
            for f in files[:10]:
                print(f'{indent}  {f}')
            if len(files) > 10:
                print(f'{indent}  ... and {len(files)-10} more files')


In [ ]:
# ================================================================
# CELL 1 — CONFIGURATION CONSTANTS
# All tunables in one place. Re-run notebook after changes.
# ================================================================

# --- Sequence Budget ---
# Layout: [ctx(396) | SEP1(1) | patches(128) | SEP2(1) | text(124)] = 650
MAX_POSITION_EMBEDDINGS = 654   # 650 + 4 for safe generation decoding
PATCH_TOKENS            = 128   # (32H/4) * (128W/8) = 8*16 = 128 patches
TEXT_TOKENS             = 12    # BOS + up to 10 word tokens + EOS
SEP_TOKENS              = 2     # SEP between ctx-patch and patch-text
CONTEXT_TOKENS          = 650 - PATCH_TOKENS - TEXT_TOKENS - SEP_TOKENS  # = 396

TOTAL_SEQ = CONTEXT_TOKENS + SEP_TOKENS + PATCH_TOKENS + TEXT_TOKENS  # = 650
assert TOTAL_SEQ == 650, f'Sequence budget: {TOTAL_SEQ} != 650'

# Position ranges (for the context-aware path)
CONTEXT_START = 0
SEP1_POS      = CONTEXT_TOKENS                                    # 396
PATCH_START   = CONTEXT_TOKENS + 1                                # 397
SEP2_POS      = CONTEXT_TOKENS + 1 + PATCH_TOKENS                # 525
TEXT_START     = CONTEXT_TOKENS + SEP_TOKENS + PATCH_TOKENS       # 526

# Loss offset: skip [ctx + SEP1 + patches + SEP2] for context-aware path
LOSS_OFFSET_CONTEXT  = CONTEXT_TOKENS + SEP_TOKENS + PATCH_TOKENS  # 526
LOSS_OFFSET_ISOLATED = PATCH_TOKENS                                 # 128

# --- Model Architecture ---
VOCAB_SIZE          = 50257     # Bengali GPT-2 BPE vocabulary
HIDDEN_SIZE         = 768
NUM_HIDDEN_LAYERS   = 12
NUM_ATTENTION_HEADS = 12
GPT2_HF_MODEL       = 'flax-community/gpt2-bengali'
VIT_HF_MODEL        = 'google/vit-base-patch16-224'
IMAGE_SIZE          = (32, 128)
PATCH_SIZE          = (4, 8)

# --- Training ---
BATCH_SIZE     = 16
LEARNING_RATE  = 1e-4           # Fixed LR, matching original train.py
TOTAL_EPOCHS   = 7              # Quick test run
GRAD_CLIP      = 1.0
USE_AMP        = True
NUM_WORKERS    = 2
PIN_MEMORY     = True

# --- Loss ---
# Context-only: L = L_ctx (standard cross-entropy)
# Dual-path disabled for now:
# MARGIN_LAMBDA  = 1     # penalty weight for margin term

# --- LR Scheduler ---
# Using CosineAnnealingWarmRestarts instead of OneCycleLR.
# Why: CosineAnnealingLR doesn't have a fixed total_steps that breaks on resume.
#      If you change TOTAL_EPOCHS mid-training, we just create a new cosine schedule
#      for the REMAINING epochs — no progress destroyed, no warmup spike.
# LR_WARMUP_STEPS = 3000       # DISABLED — no scheduler
# LR_MIN          = 3e-7       # DISABLED — no scheduler


# --- Dataset / Context ---
# ARTICLE_BARRIER: True  = context stops at article (line_id) boundaries
#                  False = context crosses articles (full cross-document)
ARTICLE_BARRIER    = True
# Budget-fill: keep adding previous words until CONTEXT_TOKENS token budget is filled
# No word-count limit — gobbles up as much context as the token budget allows
TEST_ARTICLE_RATIO = 0.1
RANDOM_SEED        = 42

# --- Paths ---
IMAGES_DIR  = '/content/data/training/images'
LABELS_FILE = '/content/data/training/labels/label.csv'

# --- Checkpoints ---
CHECKPOINT_DIR       = 'checkpoints_context_v2'
CHECKPOINT_LOAD_PATH = 'auto'  # 'auto' = find latest checkpoint in Drive/local, None = fresh start
FINAL_MODEL_PATH     = 'context_aware_v2_final.pth'
STATS_FILE           = 'training_stats/context_v2_stats.csv'
PRETRAINED_MODEL_PATH = None  # e.g. 'final_model_word_wo_pretrain_6.pth'

# ----------------------------------------------------------------
print('=' * 60)
print('SEQUENCE BUDGET (context-aware path)')
print('=' * 60)
print(f'  context : {CONTEXT_TOKENS:4d} tokens  pos {CONTEXT_START}--{SEP1_POS-1}')
print(f'  SEP1    :    1 token   pos {SEP1_POS}')
print(f'  patches : {PATCH_TOKENS:4d} tokens  pos {PATCH_START}--{SEP2_POS-1}')
print(f'  SEP2    :    1 token   pos {SEP2_POS}')
print(f'  text    : {TEXT_TOKENS:4d} tokens  pos {TEXT_START}--{TEXT_START+TEXT_TOKENS-1}')
print(f'  total   : {TOTAL_SEQ} positions (pos emb table: {MAX_POSITION_EMBEDDINGS})')
print()
print(f'  Loss offset (ctx path): logits[..., {LOSS_OFFSET_CONTEXT}:-1, :]')
print(f'  Loss offset (iso path): logits[..., {LOSS_OFFSET_ISOLATED}:-1, :]')
print()
print('TRAINING CONFIG:')
print(f'  LR = {LEARNING_RATE}, Total epochs = {TOTAL_EPOCHS}')
print(f'  Scheduler = None (fixed LR)')
print(f'  Loss: L_ctx (standard cross-entropy, context-only)')
print(f'  ARTICLE_BARRIER = {ARTICLE_BARRIER}')
print('=' * 60)


In [ ]:
# ================================================================
# CELL 2 — INSTALL DEPENDENCIES (Colab)
# Comment out if running locally with deps already installed.
# ================================================================

!pip install -q transformers torch torchvision pillow pandas scikit-learn tqdm

print('Skip install if packages already present.')


In [ ]:
# ================================================================
# CELL 3 — IMPORTS AND DEVICE SETUP
# ================================================================

import os, re, shutil
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from dataclasses import dataclass
from typing import Optional, Union, List, Tuple, Dict, Any
from torch.utils.data import Dataset, DataLoader
import tqdm

from transformers import AutoTokenizer, AutoImageProcessor
from transformers.models.vit.modeling_vit import ViTPatchEmbeddings
from transformers.models.gpt2.modeling_gpt2 import GPT2Block, GPT2Model

# ----------------------------------------------------------------
# SEED — single definition, all frameworks
# ----------------------------------------------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# ----------------------------------------------------------------
# 4D causal mask — always use SDPA format
# ----------------------------------------------------------------
try:
    from transformers.modeling_attn_mask_utils import _prepare_4d_causal_attention_mask_for_sdpa
except ImportError:
    def _prepare_4d_causal_attention_mask_for_sdpa(
        attention_mask, input_shape, inputs_embeds, past_key_values_length
    ):
        batch_size, seq_length = input_shape
        dtype, device = inputs_embeds.dtype, inputs_embeds.device
        full_seq = seq_length + past_key_values_length
        causal = torch.triu(
            torch.ones(seq_length, full_seq, dtype=torch.bool, device=device),
            diagonal=past_key_values_length + 1,
        ).unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_length, full_seq)
        if attention_mask is not None:
            pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
            causal = causal | pad
        return torch.zeros_like(causal, dtype=dtype).masked_fill_(
            causal, torch.finfo(dtype).min
        )

# ----------------------------------------------------------------
# DEVICE + AMP DTYPE SELECTION
# ----------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

# BF16 preferred: same exponent range as FP32, no overflow at 65504,
# GradScaler not needed. FP16 fallback for older GPUs (T4, etc).
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok  = torch.cuda.is_bf16_supported()
    print(f"GPU    : {gpu_name}")
    print(f"VRAM   : {gpu_vram:.1f} GB")
    print(f"BF16   : {'supported' if bf16_ok else 'NOT supported (using FP16 + GradScaler)'}")
    torch.backends.cudnn.benchmark = True
    AMP_DTYPE = torch.bfloat16 if bf16_ok else torch.float16
else:
    AMP_DTYPE = torch.bfloat16  # CPU always supports BF16
    bf16_ok = True

# GradScaler only needed for FP16 (BF16 doesn't need loss scaling)
USE_GRAD_SCALER = USE_AMP and (AMP_DTYPE == torch.float16)

print(f"PyTorch: {torch.__version__}")
print(f"AMP    : {AMP_DTYPE}  |  GradScaler: {'enabled' if USE_GRAD_SCALER else 'disabled'}")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


In [ ]:
# ================================================================
# CELL 4 — DTrOCRConfig
# Fixed: attn_implementation='sdpa', max_position_embeddings=654
# ================================================================

class DTrOCRConfig:
    def __init__(
        self,
        gpt2_hf_model: str   = GPT2_HF_MODEL,
        vit_hf_model: str    = VIT_HF_MODEL,
        vocab_size: int      = VOCAB_SIZE,
        max_position_embeddings: int = MAX_POSITION_EMBEDDINGS,  # 654 (650+4 headroom)
        hidden_size: int     = HIDDEN_SIZE,
        num_hidden_layers: int = NUM_HIDDEN_LAYERS,
        num_attention_heads: int = NUM_ATTENTION_HEADS,
        patch_size: Tuple    = PATCH_SIZE,
        image_size: Tuple    = IMAGE_SIZE,
        num_channels: int    = 3,
        resid_pdrop: float   = 0.1,
        embd_pdrop: float    = 0.1,
        attn_pdrop: float    = 0.1,
        layer_norm_epsilon: float = 1e-5,
        attn_implementation: str  = 'sdpa',  # FIX: was 'eager', now 'sdpa' matches mask format
        max_context_length: int   = CONTEXT_TOKENS,  # 396
    ):
        self.gpt2_hf_model           = gpt2_hf_model
        self.vit_hf_model            = vit_hf_model
        self.hidden_size             = hidden_size
        self.num_hidden_layers       = num_hidden_layers
        self.num_attention_heads     = num_attention_heads
        self.patch_size              = patch_size
        self.image_size              = image_size
        self.num_channels            = num_channels
        self.vocab_size              = vocab_size
        self.max_position_embeddings = max_position_embeddings
        self.resid_pdrop             = resid_pdrop
        self.embd_pdrop              = embd_pdrop
        self.attn_pdrop              = attn_pdrop
        self.layer_norm_epsilon      = layer_norm_epsilon
        self._attn_implementation    = attn_implementation
        self.max_context_length      = max_context_length

        # Required by HuggingFace GPT2Block
        self.n_inner                         = None
        self.scale_attn_weights              = True
        self.scale_attn_by_inverse_layer_idx = False
        self.reorder_and_upcast_attn         = False
        self.add_cross_attention             = False
        self.activation_function             = 'gelu_new'

    @property
    def image_embedding_length(self) -> int:
        """Number of ViT patch tokens: (H/ph) * (W/pw) = 128."""
        return int(
            (self.image_size[0] / self.patch_size[0]) *
            (self.image_size[1] / self.patch_size[1])
        )

    @property
    def text_max_length(self) -> int:
        return TEXT_TOKENS

    def __repr__(self):
        return (
            f'DTrOCRConfig(\n'
            f'  max_context_length = {self.max_context_length}\n'
            f'  image_tokens       = {self.image_embedding_length}\n'
            f'  text_max_length    = {self.text_max_length}\n'
            f'  max_positions      = {self.max_position_embeddings}\n'
            f'  hidden_size        = {self.hidden_size}\n'
            f'  num_layers         = {self.num_hidden_layers}\n'
            f'  vocab_size         = {self.vocab_size}\n'
            f'  attn_impl          = {self._attn_implementation}\n'
            f')'
        )

config = DTrOCRConfig()
print(config)
assert config.image_embedding_length == PATCH_TOKENS, f'Patch count: {config.image_embedding_length}'
print('Config OK')


In [ ]:
# ================================================================
# CELL 5 — DATA CLASSES
# ================================================================

@dataclass
class DTrOCRModelOutput:
    hidden_states: torch.FloatTensor
    past_key_values: Optional[Any]

@dataclass
class DTrOCRLMHeadModelOutput:
    logits: torch.FloatTensor
    loss: Optional[torch.FloatTensor]            = None
    accuracy: Optional[torch.FloatTensor]        = None
    past_key_values: Optional[Any]               = None
    per_sample_loss: Optional[torch.FloatTensor] = None  # [B]

@dataclass
class DTrOCRProcessorOutput:
    pixel_values:   Optional[torch.FloatTensor]            = None
    input_ids:      Optional[Union[torch.LongTensor, list]] = None
    attention_mask: Optional[Union[torch.FloatTensor, list]] = None
    labels:         Optional[Union[torch.LongTensor, list]] = None
    context_ids:    Optional[Union[torch.LongTensor, list]] = None
    context_mask:   Optional[Union[torch.FloatTensor, list]] = None

print('Data classes defined.')


In [ ]:
# ================================================================
# CELL 6 — DTrOCRProcessor
# FIX: labels use -100 for padding (ignore_index), not bos_token_id
# FIX: removed manual BOS/EOS string concat — build_inputs_with_special_tokens handles it
# ================================================================

def _modified_build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
    bos = [self.bos_token_id] if self.add_bos_token else []
    eos = [self.eos_token_id] if self.add_eos_token else []
    output = bos + token_ids_0 + eos
    if token_ids_1 is None:
        return output
    return output + bos + token_ids_1


class DTrOCRProcessor:
    def __init__(self, config: DTrOCRConfig, add_bos_token: bool = False, add_eos_token: bool = False):
        self.max_context_length = config.max_context_length
        text_max_length = config.text_max_length

        self.vit_processor = AutoImageProcessor.from_pretrained(
            config.vit_hf_model,
            size={'height': config.image_size[0], 'width': config.image_size[1]},
            use_fast=False,
        )
        self.tokeniser = AutoTokenizer.from_pretrained(
            config.gpt2_hf_model,
            add_bos_token=add_bos_token,
            model_max_length=text_max_length,
            use_fast=True,
        )
        # FIX: use eos_token as pad_token (NOT bos_token) to reduce contamination
        self.tokeniser.pad_token = self.tokeniser.eos_token
        self.tokeniser.add_eos_token = add_eos_token
        # Ensure bos_token is defined (GPT-2 may leave it as None)
        if self.tokeniser.bos_token is None:
            self.tokeniser.bos_token = self.tokeniser.eos_token
        self.tokeniser.build_inputs_with_special_tokens = (
            _modified_build_inputs_with_special_tokens.__get__(self.tokeniser)
        )
        self.add_bos_token = add_bos_token
        self.add_eos_token = add_eos_token

    def _encode_context(self, context_text: Optional[str]):
        """
        Encode context as left-padded tensor of shape [1, max_context_length].
        Left-padding: most-recent words at highest positions (closest to patches).
        """
        L      = self.max_context_length
        pad_id = self.tokeniser.pad_token_id

        if context_text and context_text.strip():
            ids = self.tokeniser.encode(context_text, add_special_tokens=False)
            ids = ids[-L:]  # keep MOST RECENT tokens if over budget
            n   = len(ids)
            ctx_ids  = [pad_id] * (L - n) + ids
            ctx_mask = [0]      * (L - n) + [1] * n
        else:
            ctx_ids  = [pad_id] * L
            ctx_mask = [0]      * L

        return (
            torch.tensor([ctx_ids],  dtype=torch.long),
            torch.tensor([ctx_mask], dtype=torch.long),
        )

    def __call__(
        self,
        images=None,
        texts=None,
        context_text: Optional[str] = None,
        return_labels: bool = False,
        input_data_format: str = 'channels_last',
        padding: Union[bool, str] = False,
        **kwargs
    ) -> DTrOCRProcessorOutput:
        # Target text — no manual BOS/EOS prepend; build_inputs_with_special_tokens handles it
        text_out = (
            self.tokeniser(texts, padding=padding, truncation=True, **kwargs)
            if texts is not None else None
        )

        # Image — ensure output is a tensor
        pixel_values = None
        if images is not None:
            img_out = self.vit_processor(images, input_data_format=input_data_format, **kwargs)
            pv = img_out['pixel_values']
            if not isinstance(pv, torch.Tensor):
                pv = torch.tensor(pv, dtype=torch.float32)
            pixel_values = pv

        # Context
        ctx_ids, ctx_mask = self._encode_context(context_text)

        # FIX: labels use -100 for padding positions so CrossEntropyLoss ignores them
        labels = None
        if texts is not None and return_labels:
            input_ids = text_out['input_ids']
            attn_mask = text_out['attention_mask']
            if isinstance(input_ids, torch.Tensor):
                labels = input_ids.clone()
                labels[attn_mask == 0] = -100
            else:
                labels = []
                for ids_row, mask_row in zip(input_ids, attn_mask):
                    labels.append([t if m == 1 else -100 for t, m in zip(ids_row, mask_row)])

        return DTrOCRProcessorOutput(
            pixel_values   = pixel_values,
            input_ids      = text_out['input_ids'] if texts is not None else None,
            attention_mask = text_out['attention_mask'] if texts is not None else None,
            labels         = labels,
            context_ids    = ctx_ids,
            context_mask   = ctx_mask,
        )

print('DTrOCRProcessor defined.')
print(f'  text_max_length    = {config.text_max_length}')
print(f'  max_context_length = {config.max_context_length}')


In [ ]:
# ================================================================
# CELL 7 — CONTEXT-AWARE DATASET
# Configurable: ARTICLE_BARRIER controls whether context crosses
#               article (line_id) boundaries or stays within one article.
# Budget-fill (token-level): no word-count limit, gobbles as much
#               context as the token budget allows.
# Context cache precomputed in __init__
# ================================================================

class ContextAwareDataset(Dataset):
    """
    Word-level dataset with budget-fill context.
    If ARTICLE_BARRIER=True, context stays within the same article (line_id).
    If ARTICLE_BARRIER=False, context crosses article boundaries freely.
    No word-count limit — fills the entire token budget.
    Context strings are precomputed and cached in __init__.
    """

    def __init__(self, images_dir: str, data_frame: 'pd.DataFrame', config: DTrOCRConfig,
                 article_barrier: bool = ARTICLE_BARRIER):
        super().__init__()
        self.images_dir = images_dir
        # Sort by image_id for correct reading order
        self.df        = data_frame.sort_values('image_id').reset_index(drop=True)
        self.image_ids = self.df['image_id'].values
        self.texts     = self.df['text'].values
        self.line_ids  = self.df['line_id'].values
        self.processor = DTrOCRProcessor(config, add_bos_token=True, add_eos_token=True)
        self.token_budget = config.max_context_length  # 396 tokens
        self.article_barrier = article_barrier

        # Precompute all context strings
        barrier_str = 'article-barrier' if self.article_barrier else 'cross-document'
        print(f'  Precomputing context cache for {len(self.df)} samples ({barrier_str})...')
        self._context_cache = [self._build_context(i) for i in range(len(self.df))]
        # Stats
        non_empty = sum(1 for c in self._context_cache if c)
        print(f'  Context cache built: {non_empty}/{len(self._context_cache)} samples have context')

    def _build_context(self, index: int) -> str:
        """
        Budget-fill context: walk backwards from index-1, adding words
        until the token budget (CONTEXT_TOKENS) is filled.
        If article_barrier=True, stop at article (line_id) boundary.
        No word-count limit — gobbles up everything the budget allows.
        """
        if index == 0:
            return ''

        current_article = self.line_ids[index]
        budget = self.token_budget
        words = []
        for i in range(index - 1, -1, -1):
            # Stop at article boundary if barrier is enabled
            if self.article_barrier and self.line_ids[i] != current_article:
                break

            candidate = str(self.texts[i])
            tokens_needed = len(self.processor.tokeniser.encode(candidate, add_special_tokens=False))
            if tokens_needed > budget:
                break
            words.insert(0, candidate)
            budget -= tokens_needed
            if budget <= 0:
                break
        return ' '.join(words)

    def _image_path(self, index: int) -> str:
        img_id = str(self.image_ids[index])
        if os.path.splitext(img_id)[1]:
            return os.path.join(self.images_dir, img_id)
        return os.path.join(self.images_dir, img_id + '.jpg')

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        path  = self._image_path(index)
        image = Image.open(path).convert('RGB')
        text  = str(self.texts[index])
        ctx   = self._context_cache[index]

        inp = self.processor(
            images         = image,
            texts          = text,
            context_text   = ctx if ctx else None,
            padding        = 'max_length',
            return_tensors = 'pt',
            return_labels  = True,
        )
        return {
            'pixel_values'  : inp.pixel_values[0],    # [3, H, W]
            'input_ids'     : inp.input_ids[0],       # [TEXT_TOKENS]
            'attention_mask': inp.attention_mask[0],   # [TEXT_TOKENS]
            'labels'        : inp.labels[0],           # [TEXT_TOKENS] with -100 for padding
            'context_ids'   : inp.context_ids[0],      # [CONTEXT_TOKENS]
            'context_mask'  : inp.context_mask[0],     # [CONTEXT_TOKENS]
        }


def split_data_context_aware(
    images_dir:  str,
    labels_file: str,
    config:      DTrOCRConfig,
    test_size:   float = TEST_ARTICLE_RATIO,
    random_seed: int   = RANDOM_SEED,
    article_barrier: bool = ARTICLE_BARRIER,
) -> Tuple[ContextAwareDataset, ContextAwareDataset]:
    """Split on article (line_id) boundaries to prevent context leakage."""
    df = pd.read_csv(labels_file)
    assert 'line_id' in df.columns, "CSV must have 'line_id' column"

    articles = df['line_id'].unique()
    n_test   = max(1, int(len(articles) * test_size))
    rng      = np.random.default_rng(random_seed)
    shuffled = rng.permutation(articles)

    test_ids  = set(shuffled[:n_test])
    train_ids = set(shuffled[n_test:])
    train_df  = df[df['line_id'].isin(train_ids)].copy()
    test_df   = df[df['line_id'].isin(test_ids)].copy()

    print(f'Articles: {len(articles)} total | {len(train_ids)} train | {len(test_ids)} test')
    print(f'Words   : {len(df)} total | {len(train_df)} train | {len(test_df)} test')
    assert len(test_ids & train_ids) == 0, 'Data leak!'

    return (
        ContextAwareDataset(images_dir, train_df, config, article_barrier=article_barrier),
        ContextAwareDataset(images_dir, test_df,  config, article_barrier=article_barrier),
    )

print(f'ContextAwareDataset defined (budget-fill, article_barrier={ARTICLE_BARRIER}, cached).')

In [ ]:
# ================================================================
# CELL 8 — DTrOCRModel (transformer backbone)
# FIX: Learned [SEP] embeddings between modalities
# FIX: Position embeddings = 654 (headroom for generation)
# FIX: SDPA attention consistently
# FIX: patch_embeddings computed externally, passed as argument
# ================================================================

class DTrOCRModel(nn.Module):
    def __init__(self, config: DTrOCRConfig):
        super().__init__()
        self.patch_embeddings     = ViTPatchEmbeddings(config)
        self.token_embedding      = nn.Embedding(config.vocab_size, config.hidden_size)
        self.positional_embedding = nn.Embedding(config.max_position_embeddings, config.hidden_size)  # 654

        # FIX: Learned separator embeddings for modality boundaries
        self.sep1_embedding = nn.Parameter(torch.randn(1, 1, config.hidden_size) * 0.02)  # ctx -> patch
        self.sep2_embedding = nn.Parameter(torch.randn(1, 1, config.hidden_size) * 0.02)  # patch -> text

        self.hidden_layers = nn.ModuleList(
            [GPT2Block(config, layer_idx=i) for i in range(config.num_hidden_layers)]
        )
        self.dropout    = nn.Dropout(config.attn_pdrop)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)
        self._attn_implementation = config._attn_implementation
        self._hidden_size = config.hidden_size

    def forward(
        self,
        input_ids:       torch.LongTensor,
        pixel_values:    Optional[torch.Tensor] = None,    # raw images (used if patch_emb is None)
        patch_emb:       Optional[torch.Tensor] = None,    # precomputed patch embeddings
        context_ids:     Optional[torch.LongTensor] = None,
        context_mask:    Optional[torch.Tensor]     = None,
        past_key_values: Optional[Tuple]            = None,
        attention_mask:  Optional[torch.Tensor]     = None,
        use_cache:       Optional[bool]             = False,
    ) -> DTrOCRModelOutput:

        dev = input_ids.device
        B   = input_ids.shape[0]

        # KV-cache state
        if past_key_values is None:
            past_length     = 0
            past_key_values = tuple([None] * len(self.hidden_layers))
        else:
            past_length = past_key_values[0][0].size(-2)

        # ---- Embeddings ----
        token_emb = self.token_embedding(input_ids)

        if past_length == 0:
            # FIX: compute patch embeddings only if not precomputed
            if patch_emb is None:
                patch_emb = self.patch_embeddings(pixel_values)

            # Expand SEP embeddings to batch size
            sep1 = self.sep1_embedding.expand(B, -1, -1)  # [B, 1, H]
            sep2 = self.sep2_embedding.expand(B, -1, -1)  # [B, 1, H]

            if context_ids is not None:
                # Context-aware: [ctx | SEP1 | patch | SEP2 | text]
                ctx_emb  = self.token_embedding(context_ids)
                combined = torch.cat([ctx_emb, sep1, patch_emb, sep2, token_emb], dim=-2)
            else:
                # Isolated: [patch | text] (no SEP tokens, no context)
                combined = torch.cat([patch_emb, token_emb], dim=-2)
        else:
            # Generation step: only new token embedding
            combined = token_emb

        seq_len = combined.shape[1]

        # ---- Positional Embeddings ----
        # FIX: clamp to max_position_embeddings-1 to prevent OOB during generation
        pos_end = min(seq_len + past_length, self.positional_embedding.num_embeddings)
        position_ids = torch.arange(
            past_length, pos_end, dtype=torch.long, device=dev
        ).unsqueeze(0)
        # If we need more positions than available, clamp the last ones
        if seq_len + past_length > self.positional_embedding.num_embeddings:
            extra = seq_len + past_length - self.positional_embedding.num_embeddings
            clamped = torch.full((1, extra), self.positional_embedding.num_embeddings - 1,
                                dtype=torch.long, device=dev)
            position_ids = torch.cat([position_ids, clamped], dim=1)

        hidden_states = combined + self.positional_embedding(position_ids)
        hidden_states = self.dropout(hidden_states)

        # ---- Attention Mask ----
        if attention_mask is not None:
            ones = lambda n: torch.ones(B, n, dtype=attention_mask.dtype, device=dev)

            if past_length == 0:
                if context_ids is not None:
                    # [ctx_mask | 1(SEP1) | ones(patch) | 1(SEP2) | text_mask]
                    full_2d = torch.cat([
                        context_mask,
                        ones(1),                    # SEP1 always attended
                        ones(patch_emb.shape[1]),   # patches always attended
                        ones(1),                    # SEP2 always attended
                        attention_mask,
                    ], dim=-1)
                else:
                    # [ones(patch) | text_mask]
                    full_2d = torch.cat([ones(patch_emb.shape[1]), attention_mask], dim=-1)
            else:
                full_2d = torch.cat([ones(past_length), attention_mask], dim=-1)

            # FIX: track padding presence without GPU->CPU sync
            has_padding = (full_2d.shape[-1] != full_2d.sum(-1).max().item()) if self._attn_implementation == 'flash_attention_2' else True
            if self._attn_implementation == 'flash_attention_2':
                attention_mask = full_2d if has_padding else None
            else:
                attention_mask = _prepare_4d_causal_attention_mask_for_sdpa(
                    attention_mask         = full_2d,
                    input_shape            = (B, seq_len),
                    inputs_embeds          = combined,
                    past_key_values_length = past_length,
                )

        # ---- Transformer Layers ----
        presents = () if use_cache else None
        for layer, layer_past in zip(self.hidden_layers, past_key_values):
            out = layer(hidden_states, layer_past=layer_past,
                        attention_mask=attention_mask, use_cache=use_cache)
            hidden_states = out[0]
            if use_cache:
                presents = presents + (out[1],)

        hidden_states = self.layer_norm(hidden_states)
        return DTrOCRModelOutput(hidden_states=hidden_states, past_key_values=presents)

    def initialise_weights(self, config: DTrOCRConfig) -> None:
        """Load pre-trained Bengali GPT-2 weights into transformer blocks + token embeddings."""
        pretrained = GPT2Model.from_pretrained(config.gpt2_hf_model, from_flax=True)
        for layer, pre_layer in zip(self.hidden_layers, pretrained.h):
            layer.load_state_dict(pre_layer.state_dict())
        self.token_embedding.load_state_dict(pretrained.wte.state_dict())
        # Copy GPT-2 positional embeddings (first 1024 positions)
        with torch.no_grad():
            n = min(pretrained.wpe.weight.shape[0], self.positional_embedding.weight.shape[0])
            self.positional_embedding.weight[:n] = pretrained.wpe.weight[:n]
        print(f'  Loaded GPT-2 Bengali weights: {config.gpt2_hf_model}')
        print(f'  Copied {n} positional embeddings from GPT-2')


print('DTrOCRModel defined (with SEP tokens, SDPA, safe positions).')


In [ ]:
# ================================================================
# CELL 9 — DTrOCRLMHeadModel
# FIX: accepts precomputed patch_emb
# FIX: loss uses ignore_index=-100 (labels already have -100 for padding)
# FIX: correct image_embedding_length = 128 (not 32)
# ================================================================

class DTrOCRLMHeadModel(nn.Module):
    def __init__(self, config: DTrOCRConfig):
        super().__init__()
        self.config              = config
        self.transformer         = DTrOCRModel(config)
        self.language_model_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.image_embedding_length = config.image_embedding_length  # 128

    def forward(
        self,
        pixel_values:    Optional[torch.Tensor] = None,
        input_ids:       Optional[torch.LongTensor] = None,
        patch_emb:       Optional[torch.Tensor] = None,     # precomputed patches
        context_ids:     Optional[torch.LongTensor] = None,
        context_mask:    Optional[torch.Tensor]     = None,
        past_key_values: Optional[Tuple]            = None,
        attention_mask:  Optional[torch.Tensor]     = None,
        use_cache:       Optional[bool]             = False,
        labels:          Optional[torch.LongTensor] = None,
    ) -> DTrOCRLMHeadModelOutput:

        out = self.transformer(
            input_ids       = input_ids,
            pixel_values    = pixel_values,
            patch_emb       = patch_emb,
            context_ids     = context_ids,
            context_mask    = context_mask,
            past_key_values = past_key_values,
            attention_mask  = attention_mask,
            use_cache       = use_cache,
        )
        logits = self.language_model_head(out.hidden_states)

        loss, accuracy, per_sample_loss = None, None, None

        if labels is not None:
            labels = labels.to(logits.device)

            # Dynamic loss offset:
            #   context present => skip [ctx(396) + SEP1(1) + patch(128) + SEP2(1)] = 526
            #   context absent  => skip [patch(128)] = 128
            if context_ids is not None and past_key_values is None:
                ctx_len = context_ids.shape[1]
                offset = ctx_len + SEP_TOKENS + self.image_embedding_length  # 396+2+128 = 526
            else:
                offset = self.image_embedding_length  # 128

            shift_logits = logits[..., offset:-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # Per-token cross-entropy with ignore_index=-100
            # FIX: padding labels are -100, so CrossEntropyLoss automatically ignores them
            loss_fn = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
            token_loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            ).view(shift_labels.shape)

            # Mask: valid positions are where labels != -100
            valid_mask = (shift_labels != -100).float()
            label_match = (shift_labels == shift_logits.argmax(dim=-1))

            per_sample_loss = (valid_mask * token_loss).sum(dim=-1) / valid_mask.sum(dim=-1).clamp(min=1)
            loss     = per_sample_loss.mean()
            accuracy = (valid_mask * label_match).sum() / valid_mask.sum().clamp(min=1)

        return DTrOCRLMHeadModelOutput(
            logits          = logits,
            loss            = loss,
            accuracy        = accuracy,
            past_key_values = out.past_key_values,
            per_sample_loss = per_sample_loss,
        )


print('DTrOCRLMHeadModel defined.')
_tmp = DTrOCRLMHeadModel(config)
print(f'Model parameters: {sum(p.numel() for p in _tmp.parameters()):,}')
del _tmp


In [ ]:
# ================================================================
# CELL 10 — UTILITY FUNCTIONS
# Context-only training: validation only runs context-aware path
# FIX: weights_only=True in all torch.load calls
# FIX: optimizer state moved to device after checkpoint load
# config_lr saved in checkpoint for exact resume
# ================================================================

def send_inputs_to_device(batch: dict, dev: str) -> dict:
    return {
        k: v.to(device=dev, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }


def run_validation(model: nn.Module, dataloader: DataLoader, dev: str):
    """
    Run validation on context-aware path only.
    Returns (ctx_loss, ctx_acc).
    """
    model.train(False)
    ctx_losses, ctx_accs = [], []

    with torch.no_grad(), torch.autocast(device_type=dev, dtype=AMP_DTYPE, enabled=USE_AMP):
        for batch in tqdm.tqdm(dataloader, desc='Validating', leave=False):
            batch = send_inputs_to_device(batch, dev)

            o_ctx = model(
                pixel_values=batch['pixel_values'], input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'], labels=batch['labels'],
                context_ids=batch['context_ids'], context_mask=batch['context_mask'],
            )
            ctx_losses.append(o_ctx.loss.item())
            ctx_accs.append(o_ctx.accuracy.item())

    model.train(True)
    avg = lambda lst: sum(lst) / len(lst) if lst else float('nan')
    return avg(ctx_losses), avg(ctx_accs)


def save_checkpoint(model, optimizer, epoch, stats_dict, chk_dir, name):
    os.makedirs(chk_dir, exist_ok=True)
    path = os.path.join(chk_dir, name)
    state_dict = model.state_dict()
    state_dict = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v
                  for k, v in state_dict.items()}
    ckpt = {
        'epoch': epoch,
        'model_state_dict': state_dict,
        'optimizer_state_dict': optimizer.state_dict(),
        'config_lr': optimizer.param_groups[0]['lr'],
        **stats_dict,
    }
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}')


def load_checkpoint(path, model, optimizer, dev):
    """Load checkpoint: model weights, optimizer state, LR."""
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(dev)
    epoch = ckpt.get('epoch', 0)
    # Restore LR if saved
    saved_lr = ckpt.get('config_lr', None)
    if saved_lr is not None:
        for g in optimizer.param_groups:
            g['lr'] = saved_lr
    print(f'Resumed from epoch {epoch}, LR={optimizer.param_groups[0]["lr"]:.2e}')
    return model, optimizer, epoch


def save_final_model(model, path):
    state_dict = model.state_dict()
    state_dict = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v
                  for k, v in state_dict.items()}
    torch.save(state_dict, path)
    print(f'Final model saved: {path}')


def load_final_model(model, path):
    sd = torch.load(path, map_location='cpu', weights_only=True)
    sd = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v
          for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    return model


print('Utility functions defined.')


---
## Debug Cells
Run each one before training to catch shape/logic errors.

In [ ]:
# # ---- DEBUG A: Budget and config consistency ----
# print('DEBUG A: Config + Budget')
# cfg = DTrOCRConfig()

# assert cfg.image_embedding_length == PATCH_TOKENS, f'Patch mismatch: {cfg.image_embedding_length}'
# total = cfg.max_context_length + SEP_TOKENS + cfg.image_embedding_length + cfg.text_max_length
# assert total == 650, f'Budget overflow: {total}'

# print(f'  context tokens : {cfg.max_context_length:4d}')
# print(f'  SEP tokens     : {SEP_TOKENS:4d}')
# print(f'  patch tokens   : {cfg.image_embedding_length:4d}')
# print(f'  text tokens    : {cfg.text_max_length:4d}')
# print(f'  total sequence : {total}')
# print(f'  pos emb table  : {cfg.max_position_embeddings} (headroom for generation)')
# print('PASSED')


In [ ]:
# # ---- DEBUG B: Processor shapes + label padding = -100 ----
# print('DEBUG B: Processor output shapes')

# cfg  = DTrOCRConfig()
# proc = DTrOCRProcessor(cfg, add_bos_token=True, add_eos_token=True)
# dummy_img = Image.fromarray(np.zeros((32, 128, 3), dtype=np.uint8))

# out = proc(images=dummy_img, texts='test', context_text='hello world',
#            padding='max_length', return_tensors='pt', return_labels=True)

# print(f'  pixel_values : {list(out.pixel_values.shape)}')
# print(f'  input_ids    : {list(out.input_ids.shape)}')
# print(f'  labels       : {list(out.labels.shape)}')
# print(f'  context_ids  : {list(out.context_ids.shape)}')
# print(f'  context_mask : {list(out.context_mask.shape)}')

# # Verify -100 padding in labels
# n_ignored = (out.labels == -100).sum().item()
# n_pad     = (out.attention_mask == 0).sum().item()
# print(f'  labels=-100 count: {n_ignored}  (should match padding count: {n_pad})')
# assert n_ignored == n_pad, f'Label padding mismatch: {n_ignored} vs {n_pad}'
# print('PASSED')


In [ ]:
# # ---- DEBUG C: Model forward pass with SEP tokens ----
# print('DEBUG C: Forward pass shapes')

# cfg = DTrOCRConfig()
# model_dbg = DTrOCRLMHeadModel(cfg).to(device)
# model_dbg.train(False)

# B = 2
# px  = torch.randn(B, 3, 32, 128).to(device)
# ids = torch.randint(0, cfg.vocab_size, (B, TEXT_TOKENS)).to(device)
# lbl = ids.clone(); lbl[:, -10:] = -100  # simulate padding
# msk = torch.ones(B, TEXT_TOKENS, dtype=torch.long).to(device); msk[:, -10:] = 0
# cid = torch.randint(0, cfg.vocab_size, (B, CONTEXT_TOKENS)).to(device)
# cmk = torch.ones(B, CONTEXT_TOKENS, dtype=torch.long).to(device); cmk[:, :20] = 0

# with torch.no_grad():
#     # Context-aware path
#     o_ctx = model_dbg(pixel_values=px, input_ids=ids, attention_mask=msk, labels=lbl,
#                       context_ids=cid, context_mask=cmk)
#     expected_ctx_seq = CONTEXT_TOKENS + SEP_TOKENS + PATCH_TOKENS + TEXT_TOKENS
#     print(f'  Ctx logits: {list(o_ctx.logits.shape)}  expect [B, {expected_ctx_seq}, vocab]')
#     assert o_ctx.logits.shape[1] == expected_ctx_seq, f'Got {o_ctx.logits.shape[1]}'

#     # Isolated path
#     o_iso = model_dbg(pixel_values=px, input_ids=ids, attention_mask=msk, labels=lbl,
#                       context_ids=None, context_mask=None)
#     expected_iso_seq = PATCH_TOKENS + TEXT_TOKENS
#     print(f'  Iso logits: {list(o_iso.logits.shape)}  expect [B, {expected_iso_seq}, vocab]')
#     assert o_iso.logits.shape[1] == expected_iso_seq

#     print(f'  Ctx loss: {o_ctx.loss.item():.4f}  acc: {o_ctx.accuracy.item():.4f}')
#     print(f'  Iso loss: {o_iso.loss.item():.4f}  acc: {o_iso.accuracy.item():.4f}')

# print('PASSED')
# del model_dbg; torch.cuda.empty_cache()


In [ ]:
# # ---- DEBUG D: Dual-path margin penalty loss ----
# print('DEBUG D: Dual-path margin penalty loss')

# cfg = DTrOCRConfig()
# model_dbg = DTrOCRLMHeadModel(cfg).to(device)
# model_dbg.train(True)

# B = 2
# px  = torch.randn(B, 3, 32, 128).to(device)
# ids = torch.randint(0, cfg.vocab_size, (B, TEXT_TOKENS)).to(device)
# lbl = ids.clone(); lbl[:, -10:] = -100
# msk = torch.ones(B, TEXT_TOKENS, dtype=torch.long).to(device); msk[:, -10:] = 0
# cid = torch.randint(0, cfg.vocab_size, (B, CONTEXT_TOKENS)).to(device)
# cmk = torch.ones(B, CONTEXT_TOKENS, dtype=torch.long).to(device)

# # Compute patch embeddings ONCE
# patch_emb = model_dbg.transformer.patch_embeddings(px)

# o_ctx = model_dbg(pixel_values=px, input_ids=ids, attention_mask=msk, labels=lbl,
#                   context_ids=cid, context_mask=cmk, patch_emb=patch_emb)
# o_iso = model_dbg(pixel_values=px, input_ids=ids, attention_mask=msk, labels=lbl,
#                   context_ids=None, context_mask=None, patch_emb=patch_emb)

# l_ctx = o_ctx.loss
# l_iso = o_iso.loss
# margin = torch.clamp(l_ctx - l_iso, min=0.0)
# total  = l_ctx + l_iso + MARGIN_LAMBDA * margin

# print(f'  L_ctx   = {l_ctx.item():.4f}')
# print(f'  L_iso   = {l_iso.item():.4f}')
# print(f'  margin  = {margin.item():.4f}  (0 if ctx helped, >0 if ctx hurt)')
# print(f'  TOTAL   = {total.item():.4f} = L_ctx + L_iso + {MARGIN_LAMBDA}*margin')

# total.backward()
# print('  backward() : OK')
# print('PASSED')
# del model_dbg; torch.cuda.empty_cache()

---
## Training
Run debug cells above first, then the two cells below.

In [ ]:
# ================================================================
# CELL 11 — TRAINING SETUP (simple, bulletproof resume)
# - Auto-detects latest checkpoint from Drive or local dir
# - Saves training config (LR, TOTAL_EPOCHS) inside checkpoint
# - On resume: uses EXACT same LR from checkpoint (no override)
# - No scheduler, no gradient accumulation
# ================================================================

if not os.path.exists(LABELS_FILE):
    raise FileNotFoundError(f'Labels file not found: {LABELS_FILE}')

cfg = DTrOCRConfig()

# --- Data ---
train_ds, val_ds = split_data_context_aware(IMAGES_DIR, LABELS_FILE, cfg)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}')

# --- Model ---
model = DTrOCRLMHeadModel(cfg)

if PRETRAINED_MODEL_PATH and os.path.exists(PRETRAINED_MODEL_PATH):
    print(f'Loading pre-trained weights: {PRETRAINED_MODEL_PATH}')
    model = load_final_model(model, PRETRAINED_MODEL_PATH)
else:
    model.transformer.initialise_weights(cfg)
    with torch.no_grad():
        model.language_model_head.weight.copy_(model.transformer.token_embedding.weight)
    print('LM head initialized from GPT-2 token embeddings (weight tying).')

model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# GradScaler
scaler = torch.amp.GradScaler(device=device, enabled=USE_GRAD_SCALER)

# --- Auto-detect latest checkpoint ---
def find_latest_checkpoint():
    """Search Drive and local dir for the latest checkpoint_epoch_N.pth"""
    import glob, re
    search_dirs = [CHECKPOINT_DIR]
    # Also check Drive
    gdrive_dir = '/content/drive/MyDrive/context_aware_v2_checkpoints'
    if os.path.isdir(gdrive_dir):
        search_dirs.append(gdrive_dir)

    best_epoch, best_path = -1, None
    for d in search_dirs:
        for f in glob.glob(os.path.join(d, 'checkpoint_epoch_*.pth')):
            m = re.search(r'checkpoint_epoch_(\d+)\.pth', f)
            if m and int(m.group(1)) > best_epoch:
                best_epoch = int(m.group(1))
                best_path = f
    return best_path, best_epoch

# --- Resume ---
loaded_epoch = 0
_resume_path = None

if CHECKPOINT_LOAD_PATH == 'auto':
    _resume_path, _found_epoch = find_latest_checkpoint()
    if _resume_path:
        print(f'Auto-detected latest checkpoint: {_resume_path} (epoch {_found_epoch})')
    else:
        print('No checkpoint found — training from scratch.')
elif CHECKPOINT_LOAD_PATH and os.path.exists(CHECKPOINT_LOAD_PATH):
    _resume_path = CHECKPOINT_LOAD_PATH
elif CHECKPOINT_LOAD_PATH:
    print(f'Checkpoint not found: {CHECKPOINT_LOAD_PATH} — training from scratch.')

if _resume_path:
    _ckpt = torch.load(_resume_path, map_location='cpu', weights_only=True)
    model.load_state_dict(_ckpt['model_state_dict'])
    optimizer.load_state_dict(_ckpt['optimizer_state_dict'])
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
    loaded_epoch = _ckpt.get('epoch', 0)

    # Restore EXACT LR from checkpoint (don't override with config)
    saved_lr = _ckpt.get('config_lr', None)
    if saved_lr is not None:
        for g in optimizer.param_groups:
            g['lr'] = saved_lr
        print(f'Resumed from epoch {loaded_epoch}, LR restored to {saved_lr:.2e}')
    else:
        # Old checkpoint without config_lr — keep optimizer's LR as-is
        print(f'Resumed from epoch {loaded_epoch}, LR from optimizer state: {optimizer.param_groups[0]["lr"]:.2e}')

    if loaded_epoch >= TOTAL_EPOCHS:
        raise ValueError(
            f'Already at epoch {loaded_epoch}, but TOTAL_EPOCHS={TOTAL_EPOCHS}. '
            f'Increase TOTAL_EPOCHS to continue training.'
        )
    del _ckpt

print(f'Fixed LR: {optimizer.param_groups[0]["lr"]:.2e} (no scheduler)')

# --- torch.compile ---
model = torch.compile(model)
print('Model compiled with torch.compile()')

# --- Stats CSV ---
os.makedirs('training_stats', exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_stats_columns = ['epoch', 'train_loss', 'train_accuracy',
                   'val_loss', 'val_accuracy', 'nan_batches', 'lr']
if not os.path.exists(STATS_FILE):
    pd.DataFrame(columns=_stats_columns).to_csv(STATS_FILE, index=False)
elif loaded_epoch > 0:
    _df = pd.read_csv(STATS_FILE)
    _df = _df[_df['epoch'] <= loaded_epoch]
    _df.to_csv(STATS_FILE, index=False)
    print(f'Stats CSV trimmed to epoch <= {loaded_epoch} ({len(_df)} rows kept)')

print()
print(f'Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Start epoch   : {loaded_epoch + 1}')
print(f'Target epoch  : {TOTAL_EPOCHS}')
print(f'Remaining     : {TOTAL_EPOCHS - loaded_epoch} epochs')
print(f'Device        : {device}')
print(f'AMP dtype     : {AMP_DTYPE}')
print(f'GradScaler    : {"enabled" if USE_GRAD_SCALER else "disabled (BF16)"}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'LR            : {optimizer.param_groups[0]["lr"]:.2e} (fixed)')
print(f'Loss          : L_ctx (standard cross-entropy, context-only)')
print(f'Scheduler     : None (fixed LR)')


In [ ]:
# ================================================================
# CELL 12 — TRAINING LOOP (simple, like original train.py)
# - Context-aware path ONLY
# - Loss: standard cross-entropy
# - Fixed LR, no scheduler
# - No gradient accumulation
# - NaN counter resets per epoch
# ================================================================

# Mount Google Drive for saving
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_SAVE_DIR = '/content/drive/MyDrive/context_aware_v2_checkpoints'
    os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)
    HAS_DRIVE = True
except ImportError:
    print('Not on Colab — skipping Google Drive mount.')
    HAS_DRIVE = False

for epoch in range(loaded_epoch, TOTAL_EPOCHS):
    model.train(True)
    ep_losses, ep_accs = [], []
    nan_count = 0

    pbar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{TOTAL_EPOCHS}')

    for step, batch in enumerate(pbar):
        batch = send_inputs_to_device(batch, device)
        optimizer.zero_grad()

        with torch.autocast(device_type=device, dtype=AMP_DTYPE, enabled=USE_AMP):
            output = model(
                pixel_values   = batch['pixel_values'],
                input_ids      = batch['input_ids'],
                attention_mask = batch['attention_mask'],
                labels         = batch['labels'],
                context_ids    = batch['context_ids'],
                context_mask   = batch['context_mask'],
            )
            loss = output.loss

        if torch.isnan(loss):
            nan_count += 1
            if nan_count <= 5:
                print(f'  [WARN] NaN loss — skipping (epoch NaN: {nan_count})')
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        ep_losses.append(loss.item())
        ep_accs.append(output.accuracy.item())

        pbar.set_postfix({
            'loss': f'{loss.item():.3f}',
            'acc': f'{output.accuracy.item():.3f}',
            'lr': f'{LEARNING_RATE:.2e}',
        })

    # ---- Epoch summary ----
    avg = lambda lst: sum(lst) / len(lst) if lst else float('nan')
    e_loss = avg(ep_losses)
    e_acc = avg(ep_accs)

    v_loss, v_acc = run_validation(model, val_loader, device)

    current_lr = LEARNING_RATE
    print(
        f'Epoch {epoch+1:3d}  '
        f'train_loss={e_loss:.4f}  train_acc={e_acc:.4f}  '
        f'val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  '
        f'nan={nan_count}  lr={current_lr:.2e}'
    )

    # Save checkpoint (includes scheduler state)
    stats = {
        'train_loss': e_loss, 'val_loss': v_loss,
        'train_acc': e_acc, 'val_acc': v_acc,
    }
    save_checkpoint(model, optimizer, epoch+1, stats, CHECKPOINT_DIR,
                    f'checkpoint_epoch_{epoch+1}.pth')

    if HAS_DRIVE:
        chk_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
        shutil.copy2(chk_path, GDRIVE_SAVE_DIR)
        print(f'  Also saved to {GDRIVE_SAVE_DIR}/')

    pd.DataFrame([{
        'epoch': epoch+1, 'train_loss': e_loss, 'train_accuracy': e_acc,
        'val_loss': v_loss, 'val_accuracy': v_acc,
        'nan_batches': nan_count, 'lr': current_lr,
    }]).to_csv(STATS_FILE, mode='a', header=False, index=False)

    # Save stats CSV to Drive every epoch
    if HAS_DRIVE:
        shutil.copy2(STATS_FILE, GDRIVE_SAVE_DIR)

# Save final model
save_final_model(model, FINAL_MODEL_PATH)
if HAS_DRIVE:
    shutil.copy2(FINAL_MODEL_PATH, GDRIVE_SAVE_DIR)
print(f'Done. Model saved to {FINAL_MODEL_PATH}')


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

def plot_training_progress(stats_file=STATS_FILE):
    if not os.path.exists(stats_file):
        print(f'No stats file found: {stats_file}')
        return

    df = pd.read_csv(stats_file)
    if len(df) == 0:
        print('No data yet — run at least 1 epoch first.')
        return

    epochs = df['epoch'].values

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Training Progress (Context-Only) — {len(df)} epochs completed', fontsize=14, fontweight='bold')

    # ---- 1. Loss ----
    ax = axes[0]
    ax.plot(epochs, df['train_loss'], 'b-o', label='Train loss', markersize=4)
    ax.plot(epochs, df['val_loss'], 'r--s', label='Val loss', markersize=4)
    ax.set_title('Loss (cross-entropy)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ---- 2. Accuracy ----
    ax = axes[1]
    ax.plot(epochs, df['train_accuracy'], 'b-o', label='Train acc', markersize=4)
    ax.plot(epochs, df['val_accuracy'], 'r--s', label='Val acc', markersize=4)
    ax.set_title('Accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ---- 3. Learning rate + NaN count ----
    ax = axes[2]
    color_lr = 'tab:orange'
    ax.plot(epochs, df['lr'], color=color_lr, marker='o', markersize=4, label='LR')
    ax.set_ylabel('Learning Rate', color=color_lr)
    ax.tick_params(axis='y', labelcolor=color_lr)
    ax.set_xlabel('Epoch')
    ax.set_title('Learning Rate & NaN Batches')
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    color_nan = 'tab:red'
    ax2.bar(epochs, df['nan_batches'], alpha=0.3, color=color_nan, label='NaN batches')
    ax2.set_ylabel('NaN count', color=color_nan)
    ax2.tick_params(axis='y', labelcolor=color_nan)

    fig.tight_layout()
    plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ---- Print verdict ----
    last = df.iloc[-1]
    print('\n' + '=' * 50)
    print('VERDICT:')
    print('=' * 50)

    if last['nan_batches'] > 0:
        print(f'  WARN: {int(last["nan_batches"])} NaN batches in last epoch')
    else:
        print(f'  GOOD: No NaN batches')

    overfit_gap = last['val_loss'] - last['train_loss']
    if overfit_gap > 0.5:
        print(f'  WARN: Overfitting gap = {overfit_gap:.4f} (val - train)')
    else:
        print(f'  GOOD: No significant overfitting (gap = {overfit_gap:.4f})')

    if len(df) >= 3:
        recent = df['val_loss'].iloc[-3:].values
        if recent[-1] < recent[0]:
            print(f'  GOOD: Val loss trending down over last 3 epochs')
        else:
            print(f'  WARN: Val loss not improving over last 3 epochs — consider stopping')
    print('=' * 50)


plot_training_progress()


In [ ]:
# ================================================================
# SAVE ALL TRAINING DATA TO GOOGLE DRIVE
# Run this after training (or anytime) to back up everything.
# ================================================================
import shutil, glob

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    raise RuntimeError('Not on Colab — Google Drive not available.')

GDRIVE_SAVE_DIR = '/content/drive/MyDrive/context_aware_v2_checkpoints'
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)

saved = []

# 1. All checkpoints
for ckpt in sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint_epoch_*.pth')):
    shutil.copy2(ckpt, GDRIVE_SAVE_DIR)
    saved.append(os.path.basename(ckpt))

# 2. Final model
if os.path.exists(FINAL_MODEL_PATH):
    shutil.copy2(FINAL_MODEL_PATH, GDRIVE_SAVE_DIR)
    saved.append(FINAL_MODEL_PATH)

# 3. Training stats CSV
if os.path.exists(STATS_FILE):
    shutil.copy2(STATS_FILE, GDRIVE_SAVE_DIR)
    saved.append(STATS_FILE)

# 4. Training progress plot
if os.path.exists('training_progress.png'):
    shutil.copy2('training_progress.png', GDRIVE_SAVE_DIR)
    saved.append('training_progress.png')

print(f'Saved {len(saved)} files to {GDRIVE_SAVE_DIR}/')
for f in saved:
    print(f'  {f}')


In [ ]:
# ================================================================
# CELL 13 — EVALUATION: BASE vs FINE-TUNED (3-way comparison)
# FIX: Base model has LM head initialized from GPT-2 (not random)
# ================================================================

if 'val_loader' not in dir():
    cfg = DTrOCRConfig()
    _, val_ds = split_data_context_aware(IMAGES_DIR, LABELS_FILE, cfg)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def evaluate_model(model_eval, dataloader, dev, use_context=False):
    model_eval.train(False)
    losses, accs = [], []
    with torch.no_grad(), torch.autocast(device_type=dev, dtype=torch.float16, enabled=USE_AMP):
        for batch in tqdm.tqdm(dataloader, desc='Eval', leave=False):
            batch = send_inputs_to_device(batch, dev)
            outputs = model_eval(
                pixel_values=batch['pixel_values'], input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'], labels=batch['labels'],
                context_ids=batch['context_ids'] if use_context else None,
                context_mask=batch['context_mask'] if use_context else None,
            )
            losses.append(outputs.loss.item())
            accs.append(outputs.accuracy.item())
    return sum(losses)/len(losses), sum(accs)/len(accs)

print('=' * 65)
print('BASELINE vs FINE-TUNED COMPARISON')
print('=' * 65)

cfg = DTrOCRConfig()

# (a) Base model — GPT-2 Bengali pretrained (with proper LM head)
print('\n[1/3] BASE model (GPT-2 Bengali, no fine-tuning)...')
base_model = DTrOCRLMHeadModel(cfg)
base_model.transformer.initialise_weights(cfg)
with torch.no_grad():
    base_model.language_model_head.weight.copy_(base_model.transformer.token_embedding.weight)
base_model.to(device)
base_model = torch.compile(base_model)
base_loss, base_acc = evaluate_model(base_model, val_loader, device, use_context=False)
print(f'  Base (isolated): loss={base_loss:.4f}  acc={base_acc*100:.2f}%')
del base_model; torch.cuda.empty_cache()

# (b) Fine-tuned — isolated
print('\n[2/3] FINE-TUNED model (isolated)...')
ft_model = DTrOCRLMHeadModel(cfg)
ft_model = load_final_model(ft_model, FINAL_MODEL_PATH)
ft_model.to(device)
ft_model = torch.compile(ft_model)
ft_iso_loss, ft_iso_acc = evaluate_model(ft_model, val_loader, device, use_context=False)
print(f'  Fine-tuned (isolated): loss={ft_iso_loss:.4f}  acc={ft_iso_acc*100:.2f}%')

# (c) Fine-tuned — with context
print('\n[3/3] FINE-TUNED model (with context)...')
ft_ctx_loss, ft_ctx_acc = evaluate_model(ft_model, val_loader, device, use_context=True)
print(f'  Fine-tuned (context):  loss={ft_ctx_loss:.4f}  acc={ft_ctx_acc*100:.2f}%')
del ft_model; torch.cuda.empty_cache()

# Summary
print('\n' + '=' * 65)
print(f'{"Model":<30} {"Loss":>10} {"Acc":>10} {"Delta":>12}')
print('-' * 65)
print(f'{"Base (GPT-2 Bengali)":<30} {base_loss:>10.4f} {base_acc*100:>9.2f}% {"---":>12}')
print(f'{"Fine-tuned (isolated)":<30} {ft_iso_loss:>10.4f} {ft_iso_acc*100:>9.2f}% {(ft_iso_acc-base_acc)*100:>+11.2f}%')
print(f'{"Fine-tuned (context)":<30} {ft_ctx_loss:>10.4f} {ft_ctx_acc*100:>9.2f}% {(ft_ctx_acc-base_acc)*100:>+11.2f}%')
print('=' * 65)
print(f'\nFine-tuning improvement (isolated): {(ft_iso_acc-base_acc)*100:+.2f}%')
print(f'Context gain over isolated:          {(ft_ctx_acc-ft_iso_acc)*100:+.2f}%')
print(f'Total improvement over base:         {(ft_ctx_acc-base_acc)*100:+.2f}%')
